In [1]:
import folium
import polars as pl
import mcr_py
import mcr_py.gtfs
import mcr_py.utils.geometa
import mcr_py.utils.footpaths
import mcr_py.command.gtfs.gtfs
import shapely

In [2]:
bounding_box = [
    (6.912789559592028, 50.95082420629814),
    (6.90965014627875, 50.94788724437913),
    (6.912218504273028, 50.94473937866948),
    (6.914029522915655, 50.944306400295005),
    (6.916940125566271, 50.946735585299706),
    (6.919886618643858, 50.94555227853371),
    (6.9237266503584465, 50.94991721673625),
    (6.926592441418052, 50.951705692912014),
    (6.925031390065072, 50.95226859486215),
    (6.923008239592917, 50.95206669530157),
    (6.923878775814018, 50.952914472192305),
    (6.921391875103581, 50.95386602541586),
    (6.919528164440663, 50.951875256766186),
    (6.912789559592028, 50.95082420629814),
]

geometa = mcr_py.utils.geometa.GeoMeta.create(
    shapely.Polygon(bounding_box), "EPSG:4326", "EPSG:4839", buffer=0
)
geometa.save("../osmtools/data/geometa.json")
mcr_py.command.gtfs.gtfs.crop_command(
    "../data/20250512/gtfs_raw/germany.zip",
    "../osmtools/data/gtfs_crop.zip",
    "01.01.1970-00:00:00",
    "01.01.2050-00:00:00",
    geometa_path="../osmtools/data/geometa.json",
)
mcr_py.command.gtfs.gtfs.clean_gtfs(
    "../osmtools/data/gtfs_crop.zip", "../osmtools/data/gtfs"
)

In [ ]:
(df_nodes, df_edges, _) = mcr_py.load_osm_walking(
    "koeln", bounding_box, "../osmtools/data/", "../osmtools/data", False
)
display(df_nodes.head())
(df_c_nodes, df_c_edges, _) = mcr_py.load_osm_cycling(
    "koeln", bounding_box, True, "../osmtools/data/", "../osmtools/data", False
)
(df_d_nodes, df_d_edges, _) = mcr_py.load_osm_driving(
    "koeln", bounding_box, "../osmtools/data/", "../osmtools/data", False
)
(df_pois) = mcr_py.load_osm_pois(
    "koeln",
    bounding_box,
    "../osmtools/data/",
    "../osmtools/data",
    False,
    nodes_to_match_df=df_nodes,  # "../osmtools/data/koeln_walking_nodes.parquet",
)

osm_id,id,lat,long
u64,u64,f64,f64
359083,0,50.950116,6.9168172
359084,1,50.95027,6.9166375
359544,2,50.948521,6.9146478
359545,3,50.948344,6.9149156
359930,4,50.946355,6.918501


In [ ]:
df_nodes.head()

In [3]:
import mcr_py.osm.graph

df_nodes = pl.read_parquet("../osmtools/data/koeln_walking_nodes.parquet")
df_c_nodes = pl.read_parquet("../osmtools/data/koeln_cycling_nodes.parquet")
df_d_nodes = pl.read_parquet("../osmtools/data/koeln_driving_nodes.parquet")
df_edges = pl.read_parquet("../osmtools/data/koeln_walking_edges.parquet")
df_c_edges = pl.read_parquet("../osmtools/data/koeln_cycling_edges.parquet")
df_d_edges = pl.read_parquet("../osmtools/data/koeln_driving_edges.parquet")
df_pois = pl.read_parquet("../osmtools/data/koeln_pois_nodes.parquet")
(df_nodes, df_edges, graph) = mcr_py.osm.graph.create_rx_graph(df_nodes, df_edges)

In [4]:
(graph_after, nodes_after, edges_after) = (
    mcr_py.osm.graph.crop_graph_to_largest_component(graph, df_nodes, df_edges)
)
paths = mcr_py.osm.graph.shortest_paths(graph_after)
df_pois_added = mcr_py.add_nearest_node_to_df(df_pois, nodes_after, "EPSG:4839")

In [5]:
df_pois = df_pois.join(
    df_nodes.select(
        pl.col("osm_id"),
        pl.col("lat").alias("nearest_lat"),
        pl.col("long").alias("nearest_lon"),
    ),
    left_on="nearest_osm_node",
    right_on="osm_id",
)


def get_coordinates(df_edges, df_nodes):
    return df_edges.join(
        df_nodes.select(
            pl.col("osm_id"),
            pl.col("lat").alias("from_lat"),
            pl.col("long").alias("from_lon"),
        ),
        left_on="source_osm",
        right_on="osm_id",
    ).join(
        df_nodes.select(
            pl.col("osm_id"),
            pl.col("lat").alias("dest_lat"),
            pl.col("long").alias("dest_lon"),
        ),
        left_on="dest_osm",
        right_on="osm_id",
    )


df_edges = get_coordinates(df_edges, df_nodes)
df_c_edges = get_coordinates(df_c_edges, df_c_nodes)
df_d_edges = get_coordinates(df_d_edges, df_d_nodes)
edges_after = get_coordinates(edges_after, nodes_after)

In [6]:
df_edges = df_edges.with_columns(
    pl.concat_list(pl.col("source").cast(pl.String), pl.col("dest").cast(pl.String))
    .list.sort()
    .list.join("")
    .is_duplicated()
    .alias("duplicated")
)
df_c_edges = df_c_edges.with_columns(
    pl.concat_list(pl.col("source").cast(pl.String), pl.col("dest").cast(pl.String))
    .list.sort()
    .list.join("")
    .is_duplicated()
    .alias("duplicated")
)
df_d_edges = df_d_edges.with_columns(
    pl.concat_list(pl.col("source").cast(pl.String), pl.col("dest").cast(pl.String))
    .list.sort()
    .list.join("")
    .is_duplicated()
    .alias("duplicated")
)

In [ ]:
m = folium.Map(location=[50.949, 6.916], zoom_start=15)
fg_walking = folium.FeatureGroup("Walking")
for edge in df_edges.iter_rows(named=True):
    folium.PolyLine(
        locations=[
            [edge["from_lat"], edge["from_lon"]],
            [edge["dest_lat"], edge["dest_lon"]],
        ],
        color="blue" if not edge["duplicated"] else "lightblue",
        popup=f"{edge['source_osm']}",
        weight=2,
        opacity=1,
    ).add_to(fg_walking)
fg_walking.add_to(m)

fg_cycling = folium.FeatureGroup("Cycling")
for edge in df_c_edges.iter_rows(named=True):
    folium.PolyLine(
        locations=[
            [edge["from_lat"], edge["from_lon"]],
            [edge["dest_lat"], edge["dest_lon"]],
        ],
        color="green" if not edge["duplicated"] else "lightgreen",
        weight=2,
        opacity=1,
    ).add_to(fg_cycling)
fg_cycling.add_to(m)
fg_driving = folium.FeatureGroup("Driving")
for edge in df_d_edges.iter_rows(named=True):
    folium.PolyLine(
        locations=[
            [edge["from_lat"], edge["from_lon"]],
            [edge["dest_lat"], edge["dest_lon"]],
        ],
        color="red" if not edge["duplicated"] else "lightcoral",
        weight=2,
        opacity=1,
    ).add_to(fg_driving)
fg_driving.add_to(m)
fg_cropped = folium.FeatureGroup("Cropped")
for edge in edges_after.iter_rows(named=True):
    folium.PolyLine(
        locations=[
            [edge["from_lat"], edge["from_lon"]],
            [edge["dest_lat"], edge["dest_lon"]],
        ],
        color="brown",
        weight=2,
        opacity=1,
    ).add_to(fg_cropped)
fg_cropped.add_to(m)

for node in nodes_after.head().iter_rows(named=True):
    folium.Marker(
        location=[node["lat"], node["long"]], popup=node["rx_node_id"]
    ).add_to(fg_cropped)


folium.LayerControl().add_to(m)
m

In [ ]:
m = folium.Map(location=[50.949, 6.916], zoom_start=15)
for node in df_pois.iter_rows(named=True):
    folium.Circle(
        location=[node["lat"], node["long"]],
        popup=f"Node {node['osm_id']}",
        radius=1,
        color="green",
    ).add_to(m)

    folium.Circle(
        location=[node["nearest_lat"], node["nearest_lon"]],
        popup=f"Node {node['nearest_osm_node']}",
        radius=1,
    ).add_to(m)

    folium.PolyLine(
        locations=[
            [node["lat"], node["long"]],
            [node["nearest_lat"], node["nearest_lon"]],
        ],
        color="red",
        weight=2,
        opacity=1,
    ).add_to(m)

for node in df_nodes.iter_rows(named=True):
    folium.Circle(
        location=[node["lat"], node["long"]],
        popup=f"Node {node['osm_id']}",
        radius=1,
    ).add_to(m)

m

In [2]:
footpaths = mcr_py.utils.footpaths.generate(
    "koeln", "../osmtools/data", "../osmtools/data/gtfs/stops.parquet", 1.4
)

{16: {'stop_name': 'Köln Ehrenfeld Körnerstr.', 'parent_station': 584990, 'stop_id': 278765, 'stop_lat': 50.94841, 'stop_lon': 6.920253, 'location_type': None, 'geometry': b'\x01\x01\x00\x00\x00\xfd.l\xcdV\xae\x1b@\x85\x99\xb6\x7feyI@', 'lat': 50.94841, 'long': 6.920253, 'nearest_node_osm_id': 31919968, 'nearest_node_distance': 97.51032358732719}, 65: {'stop_name': 'Köln Ehrenfeld Liebigstr.', 'parent_station': 108235, 'stop_id': 235011, 'stop_lat': 50.951736, 'stop_lon': 6.925193, 'location_type': None, 'geometry': b'\x01\x01\x00\x00\x00\xd8\xf35\xcbe\xb3\x1b@\x8269|\xd2yI@', 'lat': 50.951736, 'long': 6.925193, 'nearest_node_osm_id': 76598885, 'nearest_node_distance': 82.1340316135559}, 103: {'stop_name': 'Köln Ehrenfeld Venloer Str./Gürtel', 'parent_station': 416145, 'stop_id': 432314, 'stop_lat': 50.94917, 'stop_lon': 6.915545, 'location_type': None, 'geometry': b'\x01\x01\x00\x00\x00\xb6\x10\xe4\xa0\x84\xa9\x1b@\x13,\x0eg~yI@', 'lat': 50.94917, 'long': 6.915545, 'nearest_node_osm_i

In [9]:
stops = pl.read_parquet("../osmtools/data/gtfs/stops.parquet")
m = folium.Map(location=[50.949, 6.916], zoom_start=15)
for node in stops.iter_rows(named=True):
    if node["stop_id"] == 278765:
        folium.Marker(
            location=[node["stop_lat"], node["stop_lon"]],
            popup=f"Node {node['stop_id']}",
            radius=5,
            color="green",
        ).add_to(m)
    else:
        folium.Circle(
            location=[node["stop_lat"], node["stop_lon"]],
            popup=f"Node {node['stop_id']}",
            radius=5,
            color="green",
        ).add_to(m)
m